#### creating a new conda environment to work with qgis.
conda create -n qgis_env

activate the conda environment and install the required packages
```
conda activate qgis_env
conda install qgis --channel conda-forge
conda install pandas geopandas geopy openpyxl
```

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import numpy as np
from geopy.distance import geodesic
from pathlib import Path
import os
import sys

In [ ]:
CURRENT_DIRECTORY =  os.getcwd()
PARENT_DIRECTORY = os.path.dirname(CURRENT_DIRECTORY)
# {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge_polygon.gpkg|layername=sultan_bridge_polygon','OVERLAY':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/seberang_perai_multipolygons.gpkg|layername=seberang_perai_multipolygons','OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge.gpkg','GRID_SIZE':None})
BASE_PATH = Path(PARENT_DIRECTORY) / "datasets/penang_maps"


print(BASE_PATH)
print(BASE_PATH / 'penang_split/sultan_bridge_polygon.gpkg|layername=sultan_bridge_polygon')
print(BASE_PATH / 'penang_split/seberang_perai_multipolygons.gpkg|layername=seberang_perai_multipolygons', 
      BASE_PATH / 'penang_maps/penang_split/sultan_bridge.gpkg')

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge_polygon.gpkg|layername=sultan_bridge_polygon
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/seberang_perai_multipolygons.gpkg|layername=seberang_perai_multipolygons /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_maps/penang_split/sultan_bridge.gpkg


### Objective
Find out the demand for ambulances per hectare of Penang

Important data points: 
- Enroute timing and location (Ambulance start location)
- At Scene timing and location (Ambulance stopped location)
- At Patient timing and location (Paramedics reached patient)
- At Destination timing and location (Reached Hospital)
- Ready to Respond timing and location (When paramedics pressed the button to indicate they are ready to respond again) May not be accurate

Less important:
- Transporting timing and location (Detected location may not be accurate)

Penang's Max and Min coordinates:
- Latitude Min: 5.12
- Latitude Max: 5.585
- Longitude Min: 100.175
- Longitude Max: 100.551

### Getting map data from openstreetmaps

Download malaysia map from here: https://download.geofabrik.de/asia/malaysia-singapore-brunei.html#

Place the malaysia-singapore-brunei-250101.osm.pbf file in /penang_ambulance/datasets directory.

#### Directory structure 
Ensure directory looks like the one below

```text
penang_ambulance
├── datasets
│   ├── Processed_EMS_Calls_Penang_2024.csv
│   ├── Processed_EMS_Calls_Penang_2024.xlsx
│   ├── malaysia-singapore-brunei-250101.osm.pbf
│   ├── penang_maps
│   ├── ambulance_locations.xlsx
│   │   ├── penang_split
└── src
    ├── ems_data_wrangling.ipynb
    ├── ems_voronoi_steps.ipynb
    ├── ambulance_calls_description.ipynb
```

Obtained by running the `tree` command. 

Windows: `https://learn.microsoft.com/en-us/windows-server/administration/windows-commands/tree`

MacOS: `https://sourabhbajaj.com/mac-setup/iTerm/tree.html`

command used: `tree ../\<directory\>`

#### The multipolygons of Penang is obtained via the following steps

Used http://nominatim.openstreetmap.org/ to get the OSM id of Penang, George town and Seberang Peria, which will allow me to get their .poly files

Working within the `penang_ambulance/datasets/penang_maps` directory

Get the polygons for Penang. OSM id for Penang found by quering a location in Penang and looking at Adminstrative Boundary (Level 4)
`curl -L -o penang.poly "https://polygons.openstreetmap.fr/get_poly.py?id=4445131"`

Then extract penang from the OSM file.
`osmium extract -p penang.poly -s complete_ways --overwrite -o penang.osm.pbf ../malaysia-singapore-brunei-250101.osm.pbf`

First obtain the whole map of Penang
```
ogr2ogr -f GPKG penang.gpkg penang.osm.pbf
ogr2ogr -f GPKG penang_3375.gpkg penang.gpkg -t_srs EPSG:3375 
ogr2ogr -f GPKG penang_split/penang_multipolygons.gpkg penang_3375.gpkg multipolygons
```

Penang is split into 2, george town and seberang perai

1. George town polygon (OSM_id: 11203697)
```
curl -L -o penang_split/george_town.poly "https://polygons.openstreetmap.fr/get_poly.py?id=11203697"
osmium extract -p penang_split/george_town.poly -s complete_ways --overwrite -o penang_split/george_town.osm.pbf penang.osm.pbf
ogr2ogr -f GPKG penang_split/george_town.gpkg penang_split/george_town.osm.pbf
# convert to EPSG:3375
ogr2ogr -f GPKG penang_split/george_town_3375.gpkg penang_split/george_town.gpkg -t_srs EPSG:3375 
# separate lines and multipolygons into 2 different files
ogr2ogr -f GPKG penang_split/george_town_lines.gpkg penang_split/george_town_3375.gpkg lines
ogr2ogr -f GPKG penang_split/george_town_multipolygons.gpkg penang_split/george_town_3375.gpkg multipolygons

rm -f penang_split/george_town.gpkg penang_split/george_town.osm.pbf penang_split/george_town.poly penang_split/george_town_3375.gpkg
```

2. Seberang Perai polygon (OSM_id: 4447619)
```
curl -L -o penang_split/seberang_perai.poly "https://polygons.openstreetmap.fr/get_poly.py?id=4447619"
osmium extract -p penang_split/seberang_perai.poly -s complete_ways --overwrite -o penang_split/seberang_perai.osm.pbf penang.osm.pbf
ogr2ogr -f GPKG penang_split/seberang_perai.gpkg penang_split/seberang_perai.osm.pbf
# convert to EPSG:3375
ogr2ogr -f GPKG penang_split/seberang_perai_3375.gpkg penang_split/seberang_perai.gpkg -t_srs EPSG:3375
# separate lines and multipolygons into 2 different files
ogr2ogr -f GPKG penang_split/seberang_perai_lines.gpkg penang_split/seberang_perai_3375.gpkg lines
ogr2ogr -f GPKG penang_split/seberang_perai_multipolygons.gpkg penang_split/seberang_perai_3375.gpkg multipolygons

rm -f penang_split/seberang_perai.gpkg penang_split/seberang_perai.osm.pbf penang_split/seberang_perai.poly penang_split/seberang_perai_3375.gpkg
```

In [3]:
# checking that the gpkg file is using the right Coordinate Reference System
geroge_town_file_path = Path("../datasets/penang_maps/penang_split/george_town_multipolygons.gpkg")
george_town_gdf = gpd.read_file(geroge_town_file_path)
# checking that CRS of geodataframe is 3375
george_town_gdf.crs

<Projected CRS: EPSG:3375>
Name: GDM2000 / Peninsula RSO
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Malaysia - West Malaysia onshore and offshore.
- bounds: (98.02, 1.13, 105.82, 7.81)
Coordinate Operation:
- name: Peninsular RSO
- method: Hotine Oblique Mercator (variant A)
Datum: Geodetic Datum of Malaysia 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

There is a big polygon in the sea between George town and Seberang Perai (Penang mainland). It needs to be removed or it would visually interfere with the Voronoi diagram used later to find the theoretical coverage of each ambulance location. 
Since the polygon size is large, I should be able to identify it by its area.

Sorting by descending area and using qgis to visually check, the polygon OSM_way_id: 1134175227 and its name: Selat Pulau Pinang. 

If using the 2024/2025 openstreetmaps version. The polygon should be the largest polygon in the george_town_multipolygons_3375.gpkg file.

In [4]:
george_town_gdf["area_m2"] = george_town_gdf.geometry.area

george_town_gdf = george_town_gdf.sort_values(by="area_m2", ascending=False)

# removing the big polygon in the sea by its osm_way_id. Which is a string instead of an int/float
george_town_gdf = george_town_gdf[george_town_gdf['osm_way_id'] != "1134175227"]

file_path = Path("../datasets/penang_maps/penang_split/george_town_multipolygons_area.gpkg")
george_town_gdf.to_file(file_path)

#### Creating a polygon for the 2 bridges of Penang
Polygon for the bridges: 
- Sultan Abdul Halim Mu'adzam Shah Bridge
- Penang Bridge

The OSM file has information of the 2 bridges of Penang, but only Penang bridge has a polygon while the second bridge does not. To have a visual representation of the ambulance coverage, working with polygons would be easier.

Observing the multipolygon layer of either Seberang Perai or George Town, we can see that the *Sultan Abdul Halim Mu'adzam Shah Bridge* is represented by lines only. While the *Penang Bridge* is represented by lines and multipolygons.

We will need to create a polygon for the *Sultan Abdul Halim Mu'adzam Shah Bridge* and extract out the polygon for *Penang Bridge*

### Working with qgis via python commands

In [5]:
import sys
import os

QGIS_PATH = '/Applications/QGIS-LTR.app/Contents/Resources/python' 

# Add the QGIS Python path to the system path
sys.path.append(QGIS_PATH)
# Add path to plugins directory (usually needed for 'processing' module)
sys.path.append(os.path.join(QGIS_PATH, 'plugins'))

# QgsApplication.setPrefixPath() tells QGIS where to look for its core C++ libraries (plugins, resources, etc.).
# The first argument is the QGIS installation prefix (usually the folder containing 'bin', 'apps', etc.)
# For your Mac example, we use the path leading up to 'Resources'.
qgis_app_prefix = '/Applications/QGIS-LTR.app/Contents/Resources' # Adjust as needed

print(f"1. Setting QGIS prefix path to: {qgis_app_prefix}")
from qgis.core import QgsApplication
QgsApplication.setPrefixPath(qgis_app_prefix, True)

# Create an instance of QgsApplication. The second argument (False) means no GUI.
# This prepares the environment for non-GUI (standalone) scripting.
qgs = QgsApplication([], False) 

# Load Providers and registries (REQUIRED)
print("2. Initializing QGIS application...")
qgs.initQgis() 
print("QGIS application initialized successfully.")


1. Setting QGIS prefix path to: /Applications/QGIS-LTR.app/Contents/Resources
2. Initializing QGIS application...
QGIS application initialized successfully.


In [6]:
# Import QGIS Libraries and Processing ---
try:
    # Now that QGIS is initialized, we can safely import these modules
    from qgis.analysis import QgsNativeAlgorithms
    import processing
    from processing.core.Processing import Processing
    from qgis.core import QgsProject, QgsProcessingFeedback, QgsVectorLayer, QgsCoordinateReferenceSystem
    from qgis.PyQt.QtCore import QVariant 
except ImportError as e:
    print(f"\n--- FATAL ERROR: MODULE IMPORT FAILED AFTER INITIALIZATION ---")
    print(f"Please double-check the QGIS_PATH and qgis_app_prefix variables.")
    print(f"Original error: {e}")
    sys.exit(1)

#### Creating a polygon of Sultan Abdul Halim Mu'adzam Shah Bridge from the lines layer

In [ ]:
# initialise the Processing Framework
Processing.initialize()
# Add the QGIS native algorithms provider
QgsApplication.processingRegistry().addProvider(QgsNativeAlgorithms())
print("Processing framework initialized.")
print("\n--- Running Buffer Algorithm ---")

# Define Input and Output Paths (MUST be valid on your system)
# input_path = '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/seberang_perai_lines.gpkg'
input_path = str(BASE_PATH / 'penang_split/seberang_perai_lines.gpkg')
# output_path = '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge_polygon.gpkg'
output_path = str(BASE_PATH / 'penang_split/sultan_bridge_polygon.gpkg')
# The unique ID of the bridge feature
BRIDGE_OSM_ID = 725050946 
# The attribute field name
ID_FIELD_NAME = 'osm_id'

# Load only the bridge feature using subset
subset_expr = f'"{ID_FIELD_NAME}" = {BRIDGE_OSM_ID}'
layer = QgsVectorLayer(f"{input_path}|layername=lines|subset={subset_expr}", "bridge_line", "ogr")

if not layer.isValid():
    raise Exception("Failed to load input layer.")

# Select only the bridge feature
layer.selectByExpression(f'"{ID_FIELD_NAME}" = \'{BRIDGE_OSM_ID}\'')
selected_ids = layer.selectedFeatureIds()

# --- Run the buffer ---
params = {
    'INPUT': layer,
    'DISTANCE': 30,
    'SEGMENTS': 5,
    'END_CAP_STYLE': 0,      # round = 0, flat = 1, square = 2
    'JOIN_STYLE': 0,         # round = 0, miter = 1, bevel = 2
    'MITER_LIMIT': 2,
    'DISSOLVE': True,
    'SEPARATE_DISJOINT': False,
    'OUTPUT': output_path
}


Processing framework initialized.

--- Running Buffer Algorithm ---
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge_polygon.gpkg


Logged warning: Duplicate provider native registered


In [12]:
print("\n--- Running Buffer Algorithm ---")
feedback = QgsProcessingFeedback()
result = processing.run("native:buffer", params, feedback=feedback)

print("\nBuffer created successfully!")
print(f"Output saved to: {result['OUTPUT']}")

# --- Cleanup ---
qgs.exitQgis()


--- Running Buffer Algorithm ---

Buffer created successfully!
Output saved to: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge_polygon.gpkg


### Using postgres to work with geospatial data (.gpkg files)
Creating a new postgres database:

`createdb -U <user> penang_ems`

Connect to the new database:

`psql -U <user> -d penang_ems`

Transform the Postgres db into a geospatial database. IMPORTANT to run this after creating the database
```
CREATE EXTENSION postgis;
CREATE EXTENSION postgis_topology;
```

Add the gpkg file of george town and seberang perai to postgres

```
ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  "penang_split/george_town_multipolygons_area.gpkg" \
  -nln george_town_multipolygons \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  george_town_multipolygons_area

ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  "penang_split/seberang_perai_multipolygons.gpkg" \
  -nln seberang_perai_multipolygons \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  multipolygons
```

The multipolygon will contain some invalid polygons. They will need to be made valid else QGIS will throw errors when working with it.

```
UPDATE george_town_multipolygons
SET geom = ST_CollectionExtract(ST_MakeValid(geom), 3)::geometry(MultiPolygon, 3375)
WHERE NOT ST_IsValid(geom);

UPDATE seberang_perai_multipolygons
SET geom = ST_CollectionExtract(ST_MakeValid(geom), 3)::geometry(MultiPolygon, 3375)
WHERE NOT ST_IsValid(geom);
```

Then export it out, replace the existing files:
the 2 gpkg files are removed. This is because exporting the 2 tables as george_town_multipolygons_area.gpkg and seberang_perai_multipolygons.gpkg would add a new layer to the files, instead of over writting them. 

```
rm -f penang_split/george_town_multipolygons_area.gpkg penang_split/seberang_perai_multipolygons.gpkg
ogr2ogr -f GPKG penang_split/george_town_multipolygons_area.gpkg \
PG:"dbname=penang_ems user=<user>" george_town_multipolygons

ogr2ogr -f GPKG penang_split/seberang_perai_multipolygons.gpkg \
PG:"dbname=penang_ems user=<user>" seberang_perai_multipolygons
```

#### Trimming the sultan bridge polygon so it does not overlap with the other land mass's polygons

In [ ]:
# remove overlapping between bridge and seberang perai
input = (BASE_PATH / "penang_split/sultan_bridge_polygon.gpkg").as_posix() + "|layername=sultan_bridge_polygon"
overlay = (BASE_PATH / "penang_split/seberang_perai_multipolygons.gpkg").as_posix() + "|layername=seberang_perai_multipolygons"
output = (BASE_PATH / "penang_split/sultan_bridge.gpkg").as_posix()
# processing.run("native:difference", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge_polygon.gpkg|layername=sultan_bridge_polygon','OVERLAY':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/seberang_perai_multipolygons.gpkg|layername=seberang_perai_multipolygons','OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge.gpkg','GRID_SIZE':None})
processing.run("native:difference", {'INPUT': input,'OVERLAY': overlay,'OUTPUT': output,'GRID_SIZE': None})

# remove overlapping between bridge and george town
input = (BASE_PATH / "penang_split/sultan_bridge.gpkg").as_posix() + "|layername=sultan_bridge"
overlay = (BASE_PATH / "penang_split/george_town_multipolygons_area.gpkg").as_posix() + "|layername=george_town_multipolygons"
output = (BASE_PATH / "penang_split/sultan_bridge_final.gpkg").as_posix()
# processing.run("native:difference", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge.gpkg|layername=sultan_bridge','OVERLAY':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/george_town_multipolygons_area.gpkg|layername=george_town_multipolygons','OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge_final.gpkg','GRID_SIZE':None})
processing.run("native:difference", {'INPUT': input,'OVERLAY': overlay,'OUTPUT': output,'GRID_SIZE': None})

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/sultan_bridge_final.gpkg'}

#### Extracting the penang bridge polygon and removing it from george town and seberang perai polygons

In [19]:
# extract penang bridge into penang_bridge.gpkg
file_path = (BASE_PATH / "penang_split" / "george_town_multipolygons_area.gpkg").as_posix()
penang_bridge_df = gpd.read_file(file_path)
penang_bridge_df = penang_bridge_df[penang_bridge_df["osm_way_id"] == "601267471"]

penang_bridge_filepath = (BASE_PATH / "penang_split" / "penang_bridge.gpkg").as_posix()
penang_bridge_df.to_file(penang_bridge_filepath, driver="GPKG")

gt_filepath = (BASE_PATH / "penang_split" / "george_town_multipolygons_area.gpkg").as_posix()
sp_filepath = (BASE_PATH / "penang_split" / "seberang_perai_multipolygons.gpkg").as_posix()
george_town_df = gpd.read_file(gt_filepath)
seberang_perai_df = gpd.read_file(sp_filepath)
# remove penang bridge polygon from the dataframe
george_town_df = george_town_df[george_town_df["osm_way_id"] != "601267471"]
seberang_perai_df = seberang_perai_df[seberang_perai_df["osm_way_id"] != "601267471"]

gt_output = (BASE_PATH / "penang_split" / "george_town.gpkg").as_posix()
sp_output = (BASE_PATH / "penang_split" / "seberang_perai.gpkg").as_posix()

george_town_df.to_file(gt_output, driver="GPKG")
seberang_perai_df.to_file(sp_output, driver="GPKG")


#### Then replace the george town and seberang perai tables in postgres, so that penang bridge polygon is removed
Login to sql:
```
DROP TABLE george_town_multipolygons;
DROP TABLE seberang_perai_multipolygons;
```
Then in normal command line:
```
ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  "penang_split/george_town.gpkg" \
  -nln george_town_multipolygons \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  george_town

ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  "penang_split/seberang_perai.gpkg" \
  -nln seberang_perai_multipolygons \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  seberang_perai
```

#### Combining the sultan and penang bridge

In [18]:
# merging the 2 bridges
layer1 = (BASE_PATH / "penang_split/penang_bridge.gpkg").as_posix() + "|layername=penang_bridge"
layer2 = (BASE_PATH / "penang_split/sultan_bridge_final.gpkg").as_posix() + "|layername=sultan_bridge_final"
output_merge = (BASE_PATH / "penang_split/bridge_combined.gpkg").as_posix()
processing.run("native:mergevectorlayers", {'LAYERS':[layer1,layer2],'CRS':QgsCoordinateReferenceSystem('EPSG:3375'),'OUTPUT':output})

# trimming the penang bridge so it does not overlap with the land masses
input_bridge = output_merge + "|layername=bridge_combined"
overlay = (BASE_PATH / "penang_split/george_town.gpkg").as_posix() + "|layername=george_town"
output_final = (BASE_PATH / "penang_split/bridge_combined_final.gpkg").as_posix()
processing.run("native:difference", {'INPUT':input,'OVERLAY':overlay,'OUTPUT':output,'GRID_SIZE':None})

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/bridge_combined_final.gpkg'}

And add the combined bridge polygon to postgres
```
ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  penang_split/bridge_combined_final.gpkg \
  -nln bridge_combined \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  bridge_combined_final
```

### Checking ambulance locations

In [20]:
excel_path = (BASE_PATH / "ambulance_locations.xlsx").as_posix()
output_path = (BASE_PATH / "ambulance_locations.gpkg").as_posix()
### unique ambulance locations as given by Dr Fahmi
## excel sheet was obtained from Dr Fahmi's google sheet

df_ambulance_location = pd.read_excel(excel_path, sheet_name="unique_locations")
gdf_ambulance_location = gpd.GeoDataFrame(
    df_ambulance_location,
    geometry = gpd.points_from_xy(
        df_ambulance_location["longitude"],
        df_ambulance_location["latitude"]
    ),
    crs = "EPSG:4326"
)
# reproject to 3375, used for malaysia (units: metres)
gdf_ambulance_location = gdf_ambulance_location.to_crs("EPSG:3375")

# save as geopackage and then add it to pgAdmin4 for SQL querying
gdf_ambulance_location.to_file(output_path, layer="ambulance_locations", driver="GPKG")


## Generating Voronoi diagrams
Need to consider the boundary of each landmass, where it is less feasible for an ambulance at George town to serve a call at Seberang Perai. 

Each land mass polygon (george town, seberang perai and penang's bridges) will have ambulance points assigned to them, ensuring that an ambulance at george town would not cross the bridge to seberang perai for a case. This could happen as Voronoi diagram assigns the closest region to each ambulance points. 

In [ ]:
# Assign ambulances to the land mass they are on 

## Original command
# processing.run("native:extractbylocation", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/ambulance_locations.gpkg|layername=ambulance_locations','PREDICATE':[0],'INTERSECT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/seberang_perai.gpkg|layername=seberang_perai','OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/ambulances_seberang_perai.gpkg'})

# processing.run("native:extractbylocation", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/ambulance_locations.gpkg|layername=ambulance_locations','PREDICATE':[0],'INTERSECT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/george_town.gpkg|layername=george_town','OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/ambulances_george_town.gpkg'})
# extract ambulance points located within Seberang Perai polygon

processing.run(
    "native:extractbylocation",
    {
        'INPUT': (BASE_PATH / "ambulance_locations.gpkg").as_posix() + "|layername=ambulance_locations",
        'PREDICATE': [0],  # intersects
        'INTERSECT': (BASE_PATH / "penang_split/seberang_perai.gpkg").as_posix() + "|layername=seberang_perai",
        'OUTPUT': (BASE_PATH / "penang_split/ambulances_seberang_perai.gpkg").as_posix()
    }
)
# extract ambulance points located within George Town polygon
processing.run(
    "native:extractbylocation",
    {
        'INPUT': (BASE_PATH / "ambulance_locations.gpkg").as_posix() + "|layername=ambulance_locations",
        'PREDICATE': [0],  # intersects
        'INTERSECT': (BASE_PATH / "penang_split/george_town.gpkg").as_posix() + "|layername=george_town",
        'OUTPUT': (BASE_PATH / "penang_split/ambulances_george_town.gpkg").as_posix()
    }
)

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/ambulances_george_town.gpkg'}

As the bridges do not have ambulances stationed there, a buffer of 6km is created to include the nearest ambulance stations of the bridge. 

In [ ]:
# Creating a buffer for the bridges

## original command
# processing.run("native:buffer", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/bridge_combined_final.gpkg|layername=bridge_combined_final','DISTANCE':6000,'SEGMENTS':5,'END_CAP_STYLE':0,'JOIN_STYLE':0,'MITER_LIMIT':2,'DISSOLVE':False,'SEPARATE_DISJOINT':False,'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/bridge_combined_buffer.gpkg'})

processing.run(
    "native:buffer",
    {
        'INPUT': (BASE_PATH / "penang_split/bridge_combined_final.gpkg").as_posix() + "|layername=bridge_combined_final",
        'DISTANCE': 6000,
        'SEGMENTS': 5,
        'END_CAP_STYLE': 0,
        'JOIN_STYLE': 0,
        'MITER_LIMIT': 2,
        'DISSOLVE': False,
        'SEPARATE_DISJOINT': False,
        'OUTPUT': (BASE_PATH / "penang_split/bridge_combined_buffer.gpkg").as_posix()
    }
)

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/bridge_combined_buffer.gpkg'}

In [ ]:
# Assign ambulance to the bridges
## original command
# processing.run("native:extractbylocation", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/ambulance_locations.gpkg|layername=ambulance_locations','PREDICATE':[0],'INTERSECT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/bridge_combined_buffer.gpkg|layername=bridge_combined_buffer','OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/ambulances_bridges.gpkg'})

processing.run(
    "native:extractbylocation",
    {
        'INPUT': (BASE_PATH / "ambulance_locations.gpkg").as_posix() + "|layername=ambulance_locations",
        'PREDICATE': [0],
        'INTERSECT': (BASE_PATH / "penang_split/bridge_combined_buffer.gpkg").as_posix() + "|layername=bridge_combined_buffer",
        'OUTPUT': (BASE_PATH / "penang_split/ambulances_bridges.gpkg").as_posix()
    }
)

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/penang_ambulance/datasets/penang_maps/penang_split/ambulances_bridges.gpkg'}

#### Using SQL to create the voronoi diagrams for each polygon
Add the 3 ambulance locations to postgres
```
ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  penang_split/ambulances_seberang_perai.gpkg \
  -nln ambulances_seberang_perai \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  ambulances_seberang_perai

ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  penang_split/ambulances_george_town.gpkg \
  -nln ambulances_george_town \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  ambulances_george_town

ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  penang_split/ambulances_bridges.gpkg \
  -nln ambulances_bridges \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  ambulances_bridges
```

#### Create the voronoi diagrams
```
DROP TABLE IF EXISTS seberang_perai_theoretical_coverage;
CREATE TABLE seberang_perai_theoretical_coverage AS
WITH voronoi AS (
  SELECT (ST_Dump(
      ST_VoronoiPolygons(ST_Collect(geom))
  )).geom AS geom
  FROM ambulances_seberang_perai
)
SELECT a.id AS ambulance_id,
	   a."ambulance call sign" as call_sign,
	   a.comments as comments,
       ST_Intersection(v.geom, b.geom) AS geom
FROM ambulances_seberang_perai a
JOIN voronoi v
  ON ST_Contains(v.geom, a.geom)
JOIN (SELECT ST_Union(geom) AS geom FROM seberang_perai_multipolygons) b
  ON ST_Intersects(v.geom, b.geom);

DROP TABLE IF EXISTS george_town_theoretical_coverage;
CREATE TABLE george_town_theoretical_coverage AS
WITH voronoi AS (
  SELECT (ST_Dump(
      ST_VoronoiPolygons(ST_Collect(geom))
  )).geom AS geom
  FROM ambulances_george_town
)
SELECT a.id AS ambulance_id,
	   a."ambulance call sign" as call_sign,
	   a.comments as comments,
       ST_Intersection(v.geom, b.geom) AS geom
FROM ambulances_george_town a
JOIN voronoi v
  ON ST_Contains(v.geom, a.geom)
JOIN (SELECT ST_Union(geom) AS geom FROM george_town_multipolygons) b
  ON ST_Intersects(v.geom, b.geom);

DROP TABLE IF EXISTS bridges_theoretical_coverage;
CREATE TABLE bridges_theoretical_coverage AS
WITH voronoi AS (
  SELECT (ST_Dump(
      ST_VoronoiPolygons(ST_Collect(geom))
  )).geom AS geom
  FROM ambulances_bridges
)
SELECT a.id AS ambulance_id,
	   a."ambulance call sign" as call_sign,
	   a.comments as comments,
       ST_Intersection(v.geom, b.geom) AS geom
FROM ambulances_bridges a
JOIN voronoi v
  ON ST_Contains(v.geom, a.geom)
JOIN (SELECT ST_Union(geom) AS geom FROM bridge_combined) b
  ON ST_Intersects(v.geom, b.geom);
```

#### Export out the table
```
ogr2ogr -f GPKG penang_split/george_town_theoretical_coverage.gpkg \
PG:"dbname=penang_ems user=<user>" george_town_theoretical_coverage

ogr2ogr -f GPKG penang_split/seberang_perai_theoretical_coverage.gpkg \
PG:"dbname=penang_ems user=<user>" seberang_perai_theoretical_coverage

ogr2ogr -f GPKG penang_split/bridges_theoretical_coverage.gpkg \
PG:"dbname=penang_ems user=<user>" bridges_theoretical_coverage
```

## Refer to the ems_data_wrangling.ipynb notebook on how the dataset was cleaned

#### Adding the cleaned incident dataset to postgres
```
ogr2ogr -f PostgreSQL \
  PG:"dbname=penang_ems user=<user>" \
  cleaned_scene_coords.gpkg \
  -nln incidents \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  scene_coordinates
```
Then select the columns of interset from the table
```
DROP TABLE IF EXISTS incidents_shorten;

CREATE TABLE incidents_shorten AS
SELECT id, "myphc number", "call creation timestamp" day_of_week, hour_of_day, is_holiday, grouped_call_signs, geom 
FROM incidents;
```

Joining incident locations with the theoretical coverage of each ambulance point

```
DROP TABLE IF EXISTS seberang_perai_theoretical_calls;

CREATE TABLE seberang_perai_theoretical_calls AS
SELECT
	c.id as id,
	c.day_of_week as day_of_week,
	c.hour_of_day as hour_of_day,
	c.is_holiday as is_holiday,
    v.call_sign AS call_sign,
	c.geom as geom
FROM incidents_shorten AS c
INNER JOIN seberang_perai_theoretical_coverage AS v
    ON ST_Intersects(v.geom, c.geom);

DROP TABLE IF EXISTS george_town_theoretical_calls;

CREATE TABLE george_town_theoretical_calls AS
SELECT
	c.id as id,
	c.day_of_week as day_of_week,
	c.hour_of_day as hour_of_day,
	c.is_holiday as is_holiday,
    v.call_sign AS call_sign,
	c.geom as geom
FROM incidents_shorten AS c
INNER JOIN george_town_theoretical_coverage AS v
    ON ST_Intersects(v.geom, c.geom);

DROP TABLE IF EXISTS bridges_theoretical_calls;

CREATE TABLE bridges_theoretical_calls AS
SELECT
	c.id as id,
	c.day_of_week as day_of_week,
	c.hour_of_day as hour_of_day,
	c.is_holiday as is_holiday,
    v.call_sign AS call_sign,
	c.geom as geom
FROM incidents_shorten AS c
INNER JOIN bridges_theoretical_coverage AS v
    ON ST_Intersects(v.geom, c.geom);
```

Then export out

```
rm -f penang_split/george_town_theoretical_calls penang_split/seberang_theoretical_calls penang_split/bridges_theoretical_calls
ogr2ogr -f CSV penang_split/seberang_theoretical_calls.csv PG:"dbname=penang_ems user=<user>" seberang_perai_theoretical_calls

ogr2ogr -f CSV penang_split/george_town_theoretical_calls.csv PG:"dbname=penang_ems user=<user>" george_town_theoretical_calls

ogr2ogr -f CSV penang_split/bridges_theoretical_calls.csv PG:"dbname=penang_ems user=<user>" bridges_theoretical_calls
```

#### Clean up the directory a little
run in the `penang_ambulance/datasets/penang_maps` directory

```
rm -f penang_split/sultan_bridge.gpkg penang_split/sultan_bridge_polygon.gpkg penang_split/sultan_bridge_final.gpkg penang_split/seberang_perai_lines.gpkg penang_split/seberang_perai_multipolygons.gpkg penang_split/george_town_lines.gpkg penang_split/george_town_multipolygons.gpkg penang_split/george_town_multipolygons_area.gpkg penang_split/bridge_combined.gpkg penang_split/bridge_combined_buffer.gpkg penang_split/penang_bridge.gpkg
```

#### Description of the ambulance calls by callsign is found in the ambulance_calls_description.ipynb

Finally the directory should be something like this

```
penang_ambulance
├── datasets
│   ├── Processed_EMS_Calls_Penang_2024.csv
│   ├── Processed_EMS_Calls_Penang_2024.xlsx
│   ├── ambulance_total_calls.csv
│   ├── calls_by_callsign.csv
│   ├── cleaned_ems_penang.csv
│   ├── malaysia-singapore-brunei-250101.osm.pbf
│   ├── penang_ambulance.qgz # this is the file for the qgis project
│   └── penang_maps
│       ├── ambulance_locations.gpkg
│       ├── ambulance_locations.xlsx
│       ├── cleaned_scene_coords.gpkg
│       ├── penang.gpkg
│       ├── penang_split
│       │   ├── ambulances_bridges.gpkg
│       │   ├── ambulances_george_town.gpkg
│       │   ├── ambulances_seberang_perai.gpkg
│       │   ├── bridge_combined_final.gpkg
│       │   ├── bridges_theoretical_calls.csv
│       │   ├── bridges_theoretical_coverage.gpkg
│       │   ├── george_town.gpkg
│       │   ├── george_town_theoretical_calls.csv
│       │   ├── george_town_theoretical_coverage.gpkg
│       │   ├── penang_multipolygons.gpkg
│       │   ├── seberang_perai.gpkg
│       │   ├── seberang_perai_theoretical_coverage.gpkg
│       │   └── seberang_theoretical_calls.csv
│       └── scene_coordinates.gpkg
└── src
    ├── ambulance_calls_description.ipynb
    ├── ems_data_wrangling.ipynb
    └── ems_voronoi_steps.ipynb
```